# vEMDestripe on Bumbarger13: interactive, source-visible comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simple-happy-802/vEMDestripe/blob/main/colab/vEMDestripe_Bumbarger_Demo.ipynb)

This executable paper companion compares **vEMDestripe**, **EMISAC**, **FECNet**, and **SSCOR** on a fixed 32-section Bumbarger13 window.

- **vEMDestripe:** live released-checkpoint inference with a 16×16 block-mean reference.
- **EMISAC:** live execution from the authors' official GPL-3.0 repository at a verified commit. The wrapper updates only Python-2-era syntax and removed APIs; the official optimizer and parameters are retained.
- **FECNet:** frozen output from the paper's one-channel adaptation and Bumbarger-specific fine-tuning.
- **SSCOR:** frozen output from the paper's seed-42 dataset-specific self-supervised run.

> Bumbarger13 has no aligned slice-wise clean target. PSNR and SSIM would be invalid and are intentionally absent.

Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Official EMISAC is CPU-based and is the slowest live stage.

## 1. Reproducibility configuration

Configured repository: `simple-happy-802/vEMDestripe`, branch `main`. The asset is expected in the `demo-v1` GitHub Release; advanced users may paste another direct ZIP URL into `ASSET_URL`. Uploading this notebook and publishing that asset are required before sharing the launch link.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, time, zipfile

GITHUB_OWNER = "simple-happy-802"
REPOSITORY, REPOSITORY_BRANCH = "vEMDestripe", "main"
RELEASE_TAG = "demo-v1"
ASSET_NAME = "vemdestripe_bumbarger_demo_z0777_z0808.zip"
ASSET_SHA256 = "b43bf084861c50398b93815edf855dfbc7a5b3aced99cae2d6d7adc149911f3e"
ASSET_URL = f"https://github.com/{GITHUB_OWNER}/{REPOSITORY}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
RUN_VEMDESTRIPE_LIVE, RUN_OFFICIAL_EMISAC = True, True
EMISAC_WORKERS = 2

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RUNTIME_ROOT = Path("/content") if IN_COLAB else Path.cwd()/"colab_demo_runtime"
PROJECT_ROOT, DOWNLOAD_ROOT = RUNTIME_ROOT/REPOSITORY, RUNTIME_ROOT/"downloads"
RESULT_ROOT = RUNTIME_ROOT/"comparison_results"
for folder in (RUNTIME_ROOT, DOWNLOAD_ROOT, RESULT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
print(f"Runtime root: {RUNTIME_ROOT}\nColab detected: {IN_COLAB}")

## 2. Get the public code and verified demo data

The archive contains aligned input, screened references, frozen FECNet/SSCOR outputs, a vEMDestripe fallback, a NIQE pristine model, and provenance. It has no EMISAC output: that is generated from the official source below.

In [ ]:
if GITHUB_OWNER.startswith("YOUR_"):
    raise ValueError("Set GITHUB_OWNER to the public repository owner before running.")
repo_url = f"https://github.com/{GITHUB_OWNER}/{REPOSITORY}.git"
if not (PROJECT_ROOT/".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch",
                    REPOSITORY_BRANCH, repo_url, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                f"{PROJECT_ROOT}[evaluation]", "h5py>=3.9",
                "ipywidgets>=8,<9", "requests>=2.31,<3",
                "pandas>=2.2,<3", "jinja2>=3.1", "joblib>=1.4,<2"], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print(f"vEMDestripe commit: {commit}")

In [ ]:
import requests

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

archive_path = DOWNLOAD_ROOT/ASSET_NAME
if not archive_path.exists():
    partial_path = archive_path.with_suffix(".zip.part")
    print(f"Downloading {ASSET_URL}")
    with requests.get(ASSET_URL, stream=True, timeout=60) as response:
        response.raise_for_status()
        with partial_path.open("wb") as stream:
            for chunk in response.iter_content(1024*1024):
                if chunk:
                    stream.write(chunk)
    if sha256(partial_path) != ASSET_SHA256:
        raise RuntimeError("Downloaded asset checksum mismatch; remove the .part file and retry.")
    partial_path.replace(archive_path)
if sha256(archive_path) != ASSET_SHA256:
    raise RuntimeError("Cached asset checksum mismatch. Remove the cached ZIP and rerun.")

extract_root = RUNTIME_ROOT/"demo_assets"
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        destination = (extract_root/member.filename).resolve()
        if not destination.is_relative_to(extract_root.resolve()):
            raise RuntimeError(f"Unsafe archive path: {member.filename}")
    archive.extractall(extract_root)
manifests = list(extract_root.rglob("manifest.json"))
if len(manifests) != 1:
    raise RuntimeError(f"Expected one manifest, found {len(manifests)}")
ASSET_ROOT = manifests[0].parent
manifest = json.loads(manifests[0].read_text(encoding="utf-8"))
bad = [relative for relative, expected in manifest["files"].items()
       if not (ASSET_ROOT/relative).is_file()
       or sha256(ASSET_ROOT/relative) != expected]
if bad:
    raise RuntimeError(f"Asset verification failed for {bad[:5]}")
print(f"Verified ZIP and {len(manifest['files'])} files in {ASSET_ROOT}")
print(json.dumps({key: manifest[key] for key in
      ("dataset", "public_source", "data_license", "demo_local_z",
       "demo_public_z", "paired_clean_ground_truth")}, indent=2))

## 3. Run vEMDestripe live

The checkpoint uses float mean fusion followed by **16×16 block averaging**. The command disables sliding-window blending for 1024×1024 input, avoiding the old cosine-edge dark border. Without a GPU, the verified released-checkpoint output is used and labelled as a fallback.

The reference pool contains 11 independently screened sections. `num_reference=16` is an upper limit, so all 11 available sections are used, matching the released Bumbarger run.

In [ ]:
import torch

LIVE_VEM_DIR = RESULT_ROOT/"vEMDestripe"
checkpoint = PROJECT_ROOT/"checkpoints"/"vemdestripe_seed42"/"best.pth"
expected = "9a01cdef1f09716fe87369b057633f9e1278e86e77cbc70eb66b75798709c44a"
if sha256(checkpoint) != expected:
    raise RuntimeError("vEMDestripe checkpoint checksum mismatch")
if RUN_VEMDESTRIPE_LIVE and torch.cuda.is_available():
    LIVE_VEM_DIR.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(PROJECT_ROOT/"infer.py"),
        "--input_dir", str(ASSET_ROOT/"input"),
        "--output_dir", str(LIVE_VEM_DIR), "--model", str(checkpoint),
        "--reference", str(ASSET_ROOT/"reference"), "--num_reference", "16",
        "--reference_mode", "fused", "--reference_preprocess", "block_mean",
        "--struct_block_size", "16", "--reference_selection", "uniform",
        "--reference_seed", "42", "--image_size", "1024",
        "--inference_size", "1024", "--no_sliding_window", "--no_z_context",
        "--no_z_postprocess", "--no_preview", "--disable_progress_bar"]
    started = time.time(); subprocess.run(command, check=True)
    print(f"Live vEMDestripe inference: {(time.time()-started)/60:.1f} min")
    VEM_SOURCE = "live released checkpoint"
else:
    LIVE_VEM_DIR = RESULT_ROOT/"vEMDestripe_fallback"
    shutil.copytree(ASSET_ROOT/"precomputed"/"vEMDestripe", LIVE_VEM_DIR, dirs_exist_ok=True)
    VEM_SOURCE = "frozen released-checkpoint fallback (GPU unavailable or live run disabled)"
    print(VEM_SOURCE)

## 4. Run official EMISAC live

This stage clones `azadis/EMISAC` at commit `7a82a17b7bbb4ca14dff2d1c1a4a018fc8cebaf5`, verifies it, applies documented Python 3/modern-library compatibility edits, and runs the authors' optimizer. Settings are sampling 4×4, windows 32×32, partitions 4×4, `up_fg=0`, `fact=1e10`, and `lm=100`. Only the same 32-section demo window is processed.

In [ ]:
LIVE_EMISAC_DIR = RESULT_ROOT/"EMISAC_official"
EMISAC_WORK_ROOT = RUNTIME_ROOT/"emisac_official_work"
if RUN_OFFICIAL_EMISAC:
    LIVE_EMISAC_DIR.mkdir(parents=True, exist_ok=True)
    command = [sys.executable,
        str(PROJECT_ROOT/"colab"/"run_official_emisac.py"), "run",
        "--input-dir", str(ASSET_ROOT/"input"),
        "--output-dir", str(LIVE_EMISAC_DIR),
        "--work-dir", str(EMISAC_WORK_ROOT),
        "--workers", str(EMISAC_WORKERS)]
    started = time.time(); subprocess.run(command, check=True)
    print(f"Official EMISAC demo run: {(time.time()-started)/60:.1f} min")
elif not sorted(LIVE_EMISAC_DIR.glob("slice_*.png")):
    raise RuntimeError("Official EMISAC output is required. Enable RUN_OFFICIAL_EMISAC.")
metadata_path = LIVE_EMISAC_DIR/"run_metadata.json"
if not metadata_path.is_file():
    raise RuntimeError("Official EMISAC provenance metadata is required; rerun the official cell.")
emisac_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
if emisac_metadata.get("official_commit") != "7a82a17b7bbb4ca14dff2d1c1a4a018fc8cebaf5":
    raise RuntimeError("Official EMISAC commit mismatch in result metadata.")
display({key: emisac_metadata[key] for key in
         ("implementation", "official_commit", "compatibility_version", "parameters", "elapsed_seconds")})
from html import escape
from IPython.display import HTML
compatibility_diff = subprocess.check_output(
    ["git", "-C", str(EMISAC_WORK_ROOT/"official_emisac_source"),
     "diff", "--", "EM_normalization.py", "normalize_partition.py"],
    text=True, encoding="utf-8")
display(HTML("<details><summary>Inspect the complete Python 3 compatibility patch</summary>"
             "<pre style='max-height:500px;overflow:auto'>" + escape(compatibility_diff) +
             "</pre></details>"))


## 5. Load aligned volumes and provenance

The loader requires identical section indices for every method and stops rather than silently compare shifted volumes.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT/"colab"))
from demo_metrics import load_demo_volumes

volumes, section_ids = load_demo_volumes(
    ASSET_ROOT, live_emisac_dir=LIVE_EMISAC_DIR,
    live_vem_dir=LIVE_VEM_DIR)
print({name: array.shape for name, array in volumes.items()})
print(f"Local sections: {section_ids[0]}–{section_ids[-1]}")
print(f"vEMDestripe source: {VEM_SOURCE}")

## 6. Interactive XY, XZ, and YZ inspection

Use the display selector to inspect **raw outputs** or **one global affine match per volume** (the paper comparison protocol). Matching does not vary from slice to slice. Metrics below always state their matching protocol. Z is enlarged 6x only for visibility; native Z is retained except for explicitly labelled cross-plane NIQE resampling.

In [ ]:
import ipywidgets as widgets
from ipywidgets import IntSlider, interact
from demo_metrics import matched_volumes, plot_xy, plot_cross_sections

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
matched, affine_transforms = matched_volumes(volumes)
display_mode = widgets.ToggleButtons(
    options=["Raw outputs", "Global affine matched"],
    value="Global affine matched", description="Display:")
display(display_mode)

@interact(z=IntSlider(min=0, max=len(section_ids)-1,
                      value=len(section_ids)//2, description="Z offset"),
          mode=display_mode)
def inspect_xy(z, mode):
    chosen = volumes if mode == "Raw outputs" else matched
    plot_xy(chosen, section_ids, z)

In [ ]:
@interact(y=IntSlider(min=32, max=volumes["Original"].shape[1]-33,
                      step=8, value=512, description="Y"), mode=display_mode)
def inspect_xz(y, mode):
    chosen = volumes if mode == "Raw outputs" else matched
    plot_cross_sections(chosen, axis="xz", position=y, z_ratio=6.0)

@interact(x=IntSlider(min=32, max=volumes["Original"].shape[2]-33,
                      step=8, value=512, description="X"), mode=display_mode)
def inspect_yz(x, mode):
    chosen = volumes if mode == "Raw outputs" else matched
    plot_cross_sections(chosen, axis="yz", position=x, z_ratio=6.0)

## 7. Native-Z brightness profiles

Lower high-frequency oscillation is desirable, but a perfectly flat curve is not the goal because anatomy changes along Z. Interpret profiles together with orthogonal views.

In [ ]:
from demo_metrics import plot_z_profiles

@interact(mode=display_mode)
def inspect_profiles(mode):
    chosen = volumes if mode == "Raw outputs" else matched
    print(mode)
    plot_z_profiles(chosen, section_ids)

## 8. Recompute metrics on the 32-section demo window

Each processed volume receives one global affine match to Original before diagnostics. `ASID_MAD`, `local_ASID`, `Z_HF_percent`, and NIQE are lower-is-better. Contrast and blur are quality-control diagnostics, not standalone rankings. These are **demo-window results**, not the paper's 200-section benchmark.

In [ ]:
import pandas as pd
from demo_metrics import evaluate_demo

demo_metrics = evaluate_demo(
    volumes, ASSET_ROOT/"bumbarger13_niqe_model.npz",
    xy_samples=8, cross_sections=6, z_ratio=6.0, border=32)
display(demo_metrics.style.format(precision=4).background_gradient(
    subset=["ASID_MAD", "local_ASID", "Z_HF_percent",
            "NIQE_XY", "NIQE_XZ", "NIQE_YZ"], cmap="RdYlGn_r"))
demo_metrics.to_csv(RESULT_ROOT/"demo_window_metrics.csv")
print(f"Saved {RESULT_ROOT/'demo_window_metrics.csv'}")

## 9. Keep historical manuscript results separate

The optional CSV records an earlier 200-section manuscript run on local Z `[777,977)`, not a rerun of this released demo configuration. Its vEMDestripe checkpoint may differ from the released 16x16 checkpoint; its EMISAC row may come from the earlier local GPU port. It is **not used for this comparison or ranking** and is hidden by default. Enable it only for historical provenance auditing.


In [ ]:
SHOW_HISTORICAL_TABLE = False
if SHOW_HISTORICAL_TABLE:
    historical = pd.read_csv(ASSET_ROOT/"paper_metrics_200_sections.csv")
    display(historical)
else:
    print("Historical table hidden. The live 32-section table above is the official-EMISAC comparison.")

## 10. Inspect every method's official source

The viewer fetches source directly from the method authors' repositories. vEMDestripe and EMISAC use exact runtime commits. FECNet/SSCOR source is shown for transparency, but their demo images are frozen paper-run outputs.

- [vEMDestripe](https://github.com/simple-happy-802/vEMDestripe)
- [EMISAC](https://github.com/azadis/EMISAC)
- [FECNet](https://github.com/KevinJ-Huang/FECNet)
- [SSCOR](https://github.com/AI-XPM-group/SSCOR)

In [ ]:
from IPython.display import Code, Markdown, display
from functools import lru_cache
import requests

source_specs = {
 "vEMDestripe": {"repo": f"{GITHUB_OWNER}/{REPOSITORY}", "commit": commit,
                  "paths": ["vemdestripe/model.py", "infer.py", "vemdestripe/loss.py", "colab/run_official_emisac.py"]},
 "EMISAC": {"repo": "azadis/EMISAC",
             "commit": "7a82a17b7bbb4ca14dff2d1c1a4a018fc8cebaf5",
             "paths": ["EM_normalization.py", "normalize_partition.py"]},
 "FECNet": {"repo": "KevinJ-Huang/FECNet",
            "commit": "fc5a9eb224c448d190919558cc87ea7b7447f7f4",
            "paths": ["FastFourierSICE/models/archs/EnhanceN_arch.py", "FastFourierSICE/test.py"]},
 "SSCOR": {"repo": "AI-XPM-group/SSCOR",
           "commit": "985479cd79bcf1359e3d9ba44bacd5f372eb2e60",
           "paths": ["models/networks.py", "restore.py", "train.py"]}}

@lru_cache(maxsize=16)
def fetch_source(method, path):
    spec = source_specs[method]
    if method == "vEMDestripe":
        return (PROJECT_ROOT/path).read_text(encoding="utf-8")
    if method == "EMISAC":
        return subprocess.check_output(
            ["git", "-C", str(EMISAC_WORK_ROOT/"official_emisac_source"),
             "show", f"{spec['commit']}:{path}"], text=True, encoding="utf-8")
    url = f"https://raw.githubusercontent.com/{spec['repo']}/{spec['commit']}/{path}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text

method_choice = widgets.Dropdown(options=list(source_specs), description="Method:")
path_choice = widgets.Dropdown(options=source_specs[method_choice.value]["paths"],
                               description="File:", layout=widgets.Layout(width="650px"))
show_all = widgets.Checkbox(value=False, description="Show complete file")
source_output = widgets.Output()

def render_source(change=None):
    with source_output:
        source_output.clear_output(wait=True)
        method, path = method_choice.value, path_choice.value
        if path not in source_specs[method]["paths"]:
            return
        spec = source_specs[method]
        try:
            lines = fetch_source(method, path).splitlines()
            selected = lines if show_all.value else lines[:200]
            blob = f"https://github.com/{spec['repo']}/blob/{spec['commit']}/{path}"
            display(Markdown(f"**{method}**: `{path}` at `{spec['commit'][:12]}` "
                             f"([complete file]({blob})); showing {len(selected)}/{len(lines)} lines."))
            display(Code("\n".join(selected), language="python"))
        except Exception as error:
            print(f"Source viewer unavailable: {error}. Use the official repository link above.")

def change_method(change):
    path_choice.options = source_specs[change["new"]]["paths"]
    render_source()

method_choice.observe(change_method, names="value")
path_choice.observe(render_source, names="value")
show_all.observe(render_source, names="value")
display(widgets.VBox([method_choice, path_choice, show_all, source_output]))
render_source()

## 11. Export this runtime

The ZIP contains live official-EMISAC output, live/fallback vEMDestripe output, and the demo-window CSV. Frozen FECNet/SSCOR images remain in the verified input asset.

In [ ]:
runtime_provenance = {
    "vEMDestripe_commit": commit, "vEMDestripe_source": VEM_SOURCE,
    "checkpoint_sha256": sha256(checkpoint), "asset_sha256": ASSET_SHA256,
    "demo_local_z": manifest["demo_local_z"],
    "metric_protocol": "one global affine match per 32-section volume",
    "EMISAC_official_commit": emisac_metadata["official_commit"],
    "source_viewer_commits": {name: spec["commit"] for name, spec in source_specs.items()},
    "baselines": {"FECNet": "frozen grayscale-adaptation paper output",
                  "SSCOR": "frozen dataset-specific seed-42 paper output"}}
(RESULT_ROOT/"runtime_provenance.json").write_text(
    json.dumps(runtime_provenance, indent=2), encoding="utf-8")
export_base = RUNTIME_ROOT/"vemdestripe_bumbarger_demo_runtime"
export_zip = Path(shutil.make_archive(str(export_base), "zip", RESULT_ROOT))
print(f"Exported {export_zip} ({export_zip.stat().st_size/1024**2:.1f} MiB)")
if IN_COLAB:
    from google.colab import files
    download_button = widgets.Button(description="Download results ZIP", icon="download")
    download_button.on_click(lambda _: files.download(str(export_zip)))
    display(download_button)
else:
    from IPython.display import FileLink
    display(FileLink(str(export_zip)))

## Interpretation checklist

1. Verify section alignment in XY before reading metrics.
2. Use XZ/YZ views and native-Z profiles for stripe suppression.
3. Use XY, contrast, and blur diagnostics for structure preservation.
4. Treat NIQE as a diagnostic, not biological ground truth.
5. Keep 32-section live results separate from historical 200-section results.
6. Record the vEMDestripe commit, fixed EMISAC commit, asset manifest, and CSV.

Data: [Open NeuroData Bumbarger13](https://neurodata.io/data/bumbarger13/), ODC-By v1.0. Preserve `DATA_LICENSE.txt` with redistributed data.